In [4]:
!pip install pyngrok
from google.colab import drive


In [5]:
from pyngrok import ngrok

# Authenticate ngrok
!ngrok config add-authtoken 2oA42R2RK20mVb5RWDp5RUqSIRJ_5H97qcdQJwpjgfcqHFiG8


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# Import libraries
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok
import tensorflow as tf
from tensorflow.keras.models import load_model
from PIL import Image
import numpy as np

# Initialize the Flask app
app = Flask(__name__)

drive.mount('/content/drive')

skin_disease='/content/drive/MyDrive/skin_disease/skin_disease_model.keras'

# Load your model
model = load_model(skin_disease)

# Image preprocessing function
def preprocess_image(image, target_size=(224, 224)):
    image = image.resize(target_size)
    image = np.array(image) / 255.0
    image = np.expand_dims(image, axis=0)
    return image

# HTML template for the result page
result_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Skin Disease Prediction Result</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(to bottom right, #6dd5ed, #2193b0);
            color: #fff;
            margin: 0;
            padding: 0;
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
        }
        .container {
            background: rgba(255, 255, 255, 0.1);
            padding: 30px;
            border-radius: 15px;
            text-align: center;
            box-shadow: 0px 4px 15px rgba(0, 0, 0, 0.2);
            max-width: 400px;
            width: 90%;
        }
        h1 {
            font-size: 2rem;
            margin-bottom: 20px;
            color: #ffffff;
        }
        .result {
            background: #fff;
            color: #000;
            padding: 20px;
            margin: 20px 0;
            border-radius: 10px;
            font-size: 1.2rem;
            font-weight: bold;
            box-shadow: 0px 3px 10px rgba(0, 0, 0, 0.3);
        }
        button {
            background: #1abc9c;
            color: #fff;
            border: none;
            padding: 10px 20px;
            border-radius: 5px;
            font-size: 1rem;
            cursor: pointer;
            transition: 0.3s;
        }
        button:hover {
            background: #16a085;
            transform: scale(1.05);
        }
        footer {
            margin-top: 20px;
            font-size: 0.8rem;
            color: rgba(255, 255, 255, 0.7);
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Prediction Result</h1>
        <div class="result">
            Predicted Disease: {{ prediction }}
        </div>
        <button onclick="window.location.href='/'">Upload Another Image</button>
        <footer>
            Powered by AI Skin Disease Classifier
        </footer>
    </div>
</body>
</html>
"""

# HTML template for the index page
index_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Skin Disease Classifier</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(to bottom right, #6a11cb, #2575fc);
            color: #fff;
            margin: 0;
            padding: 0;
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
        }
        .container {
            background: rgba(255, 255, 255, 0.1);
            padding: 40px;
            border-radius: 15px;
            text-align: center;
            box-shadow: 0px 4px 15px rgba(0, 0, 0, 0.2);
            max-width: 400px;
            width: 90%;
        }
        h1 {
            font-size: 2rem;
            margin-bottom: 20px;
            color: #ffffff;
        }
        form {
            display: flex;
            flex-direction: column;
            align-items: center;
        }
        input[type="file"] {
            margin-bottom: 20px;
            padding: 10px;
            background: #fff;
            color: #000;
            border-radius: 5px;
            border: none;
            font-size: 1rem;
            cursor: pointer;
        }
        input[type="submit"] {
            background: #1abc9c;
            color: #fff;
            border: none;
            padding: 10px 20px;
            border-radius: 5px;
            font-size: 1rem;
            cursor: pointer;
            transition: 0.3s;
        }
        input[type="submit"]:hover {
            background: #16a085;
            transform: scale(1.05);
        }
        footer {
            margin-top: 20px;
            font-size: 0.8rem;
            color: rgba(255, 255, 255, 0.7);
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Upload an Image</h1>
        <form action="/predict" method="post" enctype="multipart/form-data">
            <input type="file" name="file" accept="image/*" required>
            <input type="submit" value="Predict">
        </form>
        <footer>
            Powered by AI Skin Disease Classifier
        </footer>
    </div>
</body>
</html>
"""

# Route for file upload and prediction
@app.route("/", methods=["GET"])
def index():
    return index_template


# Mapping model output indices to actual disease names
disease_map = {
    0: "Cellulitis",
    1: "Impetigo",
    2: "Athlete's Foot",
    3: "Nail Fungus",
    4: "Ringworm",
    5: "Larva Migrans",
    6: "Chickenpox",
    7: "Shingles"
}

@app.route("/predict", methods=["POST"])
def predict():
    if "file" not in request.files:
        return jsonify({"error": "No file uploaded"}), 400
    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No selected file"}), 400

    try:
        image = Image.open(file)
        processed_image = preprocess_image(image)
        prediction = model.predict(processed_image)

        # Get the predicted class and map it to a disease name
        disease_class = np.argmax(prediction, axis=1)[0]
        disease_label = disease_map.get(disease_class, "Unknown")

        return render_template_string(result_template, prediction=disease_label)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(" * ngrok URL:", public_url)

# Run the app
app.run(port=5000)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 * ngrok URL: NgrokTunnel: "https://c5a8-34-23-220-24.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:07:59] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:08:00] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:08:17] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:26:50] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:27:37] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:27:51] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step


INFO:werkzeug:127.0.0.1 - - [07/May/2025 07:28:07] "POST /predict HTTP/1.1" 200 -
